In [1]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.cluster import KMeans
from sklearn.model_selection import KFold
import os
import gc
from tqdm.auto import tqdm

# ==========================================
# 0. GLOBAL CONFIGURATION
# ==========================================
SEED = 8503
BASE_PATH = "./"

# ==========================================
# 1. METRICS & HELPERS
# ==========================================
def _clip01(x: float) -> float:
    return float(np.minimum(np.maximum(x, 0.0), 1.0))

def weighted_rmse_score(y_target, y_pred, w) -> float:
    denom = np.sum(w * y_target ** 2)
    if denom == 0.0: return 0.0
    ratio = np.sum(w * (y_target - y_pred) ** 2) / denom
    clipped = _clip01(ratio)
    return float(np.sqrt(1.0 - clipped))

class LgbTqdmCallback:
    def __init__(self, max_iter, desc="Boosting"):
        self.pbar = tqdm(total=max_iter, desc=desc, leave=False)
    def __call__(self, env):
        self.pbar.update(1)
        if env.evaluation_result_list:
            score = env.evaluation_result_list[0][2]
            self.pbar.set_postfix({'val_rmse': f'{score:.5f}'})
    def __del__(self):
        self.pbar.close()

def get_predictive_clusters(X, y, codes, n_clusters=5):
    """Scout model to profile stocks and group them."""
    base_model = lgb.LGBMRegressor(n_estimators=150, learning_rate=0.1, n_jobs=-1, random_state=SEED, verbosity=-1)
    base_model.fit(X, y)
    preds = base_model.predict(X)
    
    profile_df = pd.DataFrame({'code': codes, 'actual': y, 'pred': preds, 'error': preds - y})
    stock_profiles = profile_df.groupby('code').agg(
        mean_actual=('actual', 'mean'), 
        mean_pred=('pred', 'mean'), 
        mean_error=('error', 'mean')
    ).fillna(0) 
    
    kmeans = KMeans(n_clusters=n_clusters, random_state=SEED, n_init='auto')
    stock_profiles['cluster_id'] = kmeans.fit_predict(stock_profiles)
    return stock_profiles['cluster_id'].to_dict()

# ==========================================
# PHASE 1: STRICT TRAINING (NO TEST DATA ALLOWED)
# ==========================================
print("\n" + "="*50)
print(" PHASE 1: TRAINING ENSEMBLE + CLUSTER FEATURES")
print("="*50)

print("Loading TRAIN data exclusively...")
try:
    train = pd.read_parquet(os.path.join(BASE_PATH, "train.parquet"))
except FileNotFoundError:
    print(f"Warning: train.parquet not found at {BASE_PATH}. Please verify the path.")

cat_cols = ['code', 'sub_code', 'sub_category']
feature_cols = [c for c in train.columns if c.startswith('feature_')]

# 1. Zero-Leakage Categorical Encoding
label_maps = {}
for col in cat_cols:
    unique_vals = train[col].astype(str).unique()
    label_maps[col] = {val: i for i, val in enumerate(unique_vals)}
    train[col] = train[col].astype(str).map(label_maps[col])

params = {
    'objective': 'regression', 'metric': 'rmse', 'boosting_type': 'gbdt',
    'learning_rate': 0.05, 'num_leaves': 31, 'feature_fraction': 0.8,
    'verbosity': -1, 'seed': SEED, 'n_jobs': -1
}
N_ESTIMATORS = 800
N_CLUSTERS = 5 
horizons = train['horizon'].unique()

train['oof_preds'] = 0.0 

# Dictionary to hold models and cluster maps for Phase 2
horizon_artifacts = {}

for h in tqdm(horizons, desc="Training Horizons"):
    train_h = train[train['horizon'] == h].copy()
    
    tqdm.write(f"  [Horizon {h}] Profiling stocks and generating clusters...")
    cluster_map = get_predictive_clusters(
        X=train_h[feature_cols], y=train_h['y_target'], codes=train_h['code'], n_clusters=N_CLUSTERS
    )
    
    train_h['cluster_id'] = train_h['code'].map(cluster_map).fillna(0).astype(int)
    
    current_features = feature_cols + cat_cols + ['cluster_id']
    current_cats = cat_cols + ['cluster_id']
    
    kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
    horizon_oof = np.zeros(len(train_h))
    horizon_experts = []
    
    fold = 1
    for train_idx, val_idx in kf.split(train_h):
        X_tr, y_tr, w_tr = train_h[current_features].iloc[train_idx], train_h['y_target'].iloc[train_idx], train_h['weight'].iloc[train_idx]
        X_va, y_va, w_va = train_h[current_features].iloc[val_idx], train_h['y_target'].iloc[val_idx], train_h['weight'].iloc[val_idx]
        
        model = lgb.LGBMRegressor(**params, n_estimators=N_ESTIMATORS)
        tqdm_cb = LgbTqdmCallback(max_iter=N_ESTIMATORS, desc=f"H{h} Fold {fold}")
        
        model.fit(
            X_tr, y_tr, sample_weight=w_tr,
            eval_set=[(X_va, y_va)], eval_sample_weight=[w_va],
            categorical_feature=current_cats,
            callbacks=[lgb.early_stopping(50, verbose=False), tqdm_cb]
        )
        tqdm_cb.pbar.close()
        
        horizon_oof[val_idx] = model.predict(X_va)
        horizon_experts.append(model)
        fold += 1
        
    h_score = weighted_rmse_score(train_h['y_target'].values, horizon_oof, train_h['weight'].values)
    tqdm.write(f"✓ Horizon {h} OOF Score: {h_score:.5f}\n")
    train.loc[train['horizon'] == h, 'oof_preds'] = horizon_oof
    
    # Save artifacts needed for inference
    horizon_artifacts[h] = {
        'cluster_map': cluster_map,
        'experts': horizon_experts,
        'features': current_features
    }
    
    gc.collect()

global_score = weighted_rmse_score(train['y_target'].values, train['oof_preds'].values, train['weight'].values)
print("\n" + "="*50)
print(f"⭐ EXACT GLOBAL OOF SCORE (Zero Leakage): {global_score:.5f}")
print("="*50)

# ==========================================
# PHASE 2: STRICT INFERENCE (LOADING TEST SET)
# ==========================================
print("\n" + "="*50)
print(" PHASE 2: INFERENCE")
print("="*50)

print("Loading TEST data...")
try:
    test = pd.read_parquet(os.path.join(BASE_PATH, "test.parquet"))
except FileNotFoundError:
    print(f"Warning: test.parquet not found at {BASE_PATH}.")

# 2. Apply strict mapping. Unseen categories map to -1.
for col in cat_cols:
    test[col] = test[col].astype(str).map(label_maps[col]).fillna(-1).astype(int)

test['y_target'] = 0.0

for h in tqdm(horizons, desc="Inferring Horizons"):
    test_h = test[test['horizon'] == h].copy()
    if len(test_h) == 0: continue
    
    artifacts = horizon_artifacts[h]
    
    # Apply horizon-specific cluster map
    test_h['cluster_id'] = test_h['code'].map(artifacts['cluster_map']).fillna(0).astype(int)
    
    expert_predictions = np.zeros(len(test_h))
    for expert in artifacts['experts']:
        expert_predictions += expert.predict(test_h[artifacts['features']])
        
    final_ensemble_preds = expert_predictions / len(artifacts['experts'])
    test.loc[test['horizon'] == h, 'y_target'] = final_ensemble_preds

# ==========================================
# 3. GLOBAL SCORING & SUBMISSION
# ==========================================
submission = test[['id', 'y_target']].copy()
submission.to_csv('submission.csv', index=False)
print("\n✅ Submission file 'submission.csv' generated securely!")

c:\Users\User\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



 PHASE 1: TRAINING ENSEMBLE + CLUSTER FEATURES
Loading TRAIN data exclusively...


Training Horizons:   0%|          | 0/4 [00:02<?, ?it/s]

  [Horizon 25] Profiling stocks and generating clusters...


Training Horizons:   0%|          | 0/4 [03:15<?, ?it/s]

...

✓ Horizon 10 OOF Score: 0.46693



Training Horizons: 100%|██████████| 4/4 [12:51<00:00, 193.00s/it]



⭐ EXACT GLOBAL OOF SCORE (Zero Leakage): 0.50658

 PHASE 2: INFERENCE
Loading TEST data...


Inferring Horizons: 100%|██████████| 4/4 [00:25<00:00,  6.47s/it]



✅ Submission file 'submission.csv' generated securely!
